# Checkpointers (Part 1) — In-Memory Short-Term Memory

This notebook extends the [Agents](../../LangChain%20Agents%20%26%20Tools/Labs/Agents.ipynb) example by adding a **checkpointer** — the mechanism that gives an agent memory across multiple messages in the same conversation thread.

## Key concepts

- **Checkpointer** – Saves a snapshot ("checkpoint") of the agent's full state after every step. When you send the next message in the same thread, the agent restores from the last checkpoint, effectively "remembering" the conversation.
- **`InMemorySaver`** – The simplest checkpointer: stores all checkpoints in RAM. Fast and easy to use, but data is lost when the Python process ends (no persistence between sessions).
- **`thread_id`** – A string that identifies a conversation thread. All messages sharing the same `thread_id` are linked together. Different `thread_id` values = separate, isolated conversations.
- **`RunnableConfig`** – A configuration dictionary passed to `.invoke()`. You put the `thread_id` inside its `"configurable"` key.
- **`explore_checkpoints()`** – A helper we write to inspect the raw checkpoint data stored by the saver.

## Mental model

Think of a checkpointer as a **save slot in a video game**. Before asking the next question, the agent automatically loads its save from the previous turn — so it knows who you are, what you asked, and what it answered.

In [ ]:
# Install the required packages.
# langchain-openai adds the ChatOpenAI model wrapper.
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata                          # Colab's secret manager
from langchain.agents import create_agent                  # Factory function that builds an agent
from langchain.messages import HumanMessage                # Represents a user's message
from langchain.tools import tool                           # Decorator for turning a function into a LangChain tool
from langchain_core.messages import BaseMessage            # Base class for all message types (used in type hints)
from langchain_core.runnables.config import RunnableConfig # Type hint for the config dict (holds thread_id etc.)
from langchain_openai import ChatOpenAI                    # OpenAI LLM integration
from langgraph.checkpoint.base import BaseCheckpointSaver  # Abstract base class for all checkpointers
from langgraph.checkpoint.memory import InMemorySaver      # RAM-based checkpointer (no persistence)
from pydantic import SecretStr                             # Wraps strings so they don't get accidentally logged
from typing import List

# Securely load the OpenAI API key from Colab secrets.
openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper: pretty-print a conversation (list of messages).
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

# Helper: inspect all checkpoints saved for a given thread.
# Each checkpoint_tuple contains the full agent state at a particular step.
def explore_checkpoints(checkpoint_saver: BaseCheckpointSaver, runnable_config: RunnableConfig):
    for checkpoint_tuple in checkpoint_saver.list(runnable_config):
        print(checkpoint_tuple)
        # channel_values holds the actual state data (messages, tool outputs, etc.)
        print(checkpoint_tuple.checkpoint["channel_values"])

In [ ]:
# --- Define the agent's tools ---
# The @tool decorator converts a regular Python function into a LangChain tool
# that the AI model can decide to call when it needs information.

@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """
    # A simple hardcoded "database" of tour stops (dictionary lookup).
    # In a real app this would query an actual API or database.
    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    # Normalize the input to lowercase before looking it up.
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    # A hardcoded lookup table for driving times between Bulgarian cities.
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    # Build the lookup key from normalized (lowercase) city names.
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")

In [ ]:
# Create the in-memory checkpointer.
# This object will store all agent state snapshots in a Python dictionary in RAM.
# Simple and fast — ideal for development and single-session demos.
checkpointer = InMemorySaver()

In [ ]:
# Build the agent and pass the checkpointer.
# The checkpointer is what enables memory — without it, every .invoke() call
# would start a fresh conversation with no recollection of earlier messages.
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key),
    tools=[lookup_tour_stop, estimate_drive_time],
    system_prompt="You are a practical live-music concierge. Use tools when they help you give a precise answer.",
    checkpointer=checkpointer  # <-- This is what gives the agent memory!
)

In [ ]:
# Define the thread configuration.
# thread_id is a unique identifier for this conversation session.
# ALL messages sent with this same config belong to the same conversation thread —
# the agent will remember everything said within it.
# Change the thread_id to start a brand new, independent conversation.
config: RunnableConfig = {
    "configurable": {
        "thread_id": "1"
    }
}

In [ ]:
# Send the first user message.
# The agent will:
#   1. Call lookup_tour_stop("Vassil Petrov & JP3") to find the venue city.
#   2. Call estimate_drive_time("Plovdiv", <venue city>) to get drive time.
#   3. Compose a helpful answer combining both tool results.
# After execution, the checkpointer saves a snapshot of the entire state.
plan_concert_trip = agent.invoke(
    input={
      "messages": [HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time.")]
    },
    config=config  # Links this call to thread "1"
)

In [ ]:
# Display the conversation so far — user message, tool calls, tool results, and the final AI reply.
print_conversation(plan_concert_trip["messages"])

In [ ]:
# Inspect the checkpoints saved after the first conversation turn.
# You'll see one checkpoint entry per agent step (model call, tool call, etc.).
# Each entry shows the full agent state at that moment in time.
explore_checkpoints(checkpointer, config)

In [ ]:
# Send a FOLLOW-UP message in the SAME thread ("1").
# Because we use the same config (same thread_id), the agent loads the previous checkpoint
# and can recall everything from the earlier conversation turn.
# The agent does NOT need to be told what was discussed — it already "remembers".
test_agent_memory = agent.invoke(
    input={
        "messages": [HumanMessage("What do you remember about me?")]
    },
    config=config  # Same thread_id = same conversation context
)

In [ ]:
# Print the conversation for the second turn.
# Notice the full message history (from both turns) is included — 
# the agent sees the entire thread each time.
print_conversation(test_agent_memory["messages"])

In [ ]:
# Inspect checkpoints again after the second turn.
# Compare the output with the first exploration — you'll see new checkpoint entries
# were added for the steps in the second .invoke() call.
explore_checkpoints(checkpointer, config)